# Multi-Tier LLM Routing Agent — FAISS + Learned Router + Semantic Cache

**Goal:** Route incoming text extraction queries to the *cheapest model that can still get it right*

This notebook builds three upgrades on top of a naive "easy → small model, hard → big model" router:

1. **A custom confidence score** — combines kNN vote margin, local density, and logistic-regression entropy, so the router knows when *not* to trust itself.
2. **A learned router** — a logistic regression trained on frozen sentence embeddings (same methodology as the gender-probing work: freeze the encoder, train a linear probe on top).
3. **A semantic cache** — near-duplicate queries skip inference entirely via FAISS similarity lookup.

**Pipeline:** `query → cache check → embed → LR predicts tier + confidence score → escalate if low-confidence → route to Qwen2.5-{0.5B, 1.5B, 3B} → extract entities → cache result`

**Stack:** Google Colab (free tier, T4 GPU), `sentence-transformers`, `faiss-cpu`, `transformers`, `bitsandbytes`, `scikit-learn`. 


## 0. Installs

Run once per Colab session. Each package is installed separately so a failure is easy to isolate.

In [ ]:
!pip install -q sentence-transformers==3.0.1
!pip install -q faiss-cpu==1.8.0
!pip install -q transformers==4.44.2 accelerate==0.33.0
!pip install -q bitsandbytes==0.43.3
!pip install -q scikit-learn==1.5.1
!pip install -q pandas==2.2.2 matplotlib==3.9.1


In [ ]:
import time, json, re, random, textwrap
import numpy as np
import pandas as pd
import faiss
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 1. Synthetic financial extraction dataset

There's no off-the-shelf dataset labeled by "query complexity," so we generate one programmatically from templates. Because *we* control generation, we get **two** free labels per sample:

- **complexity tier** (0 = easy, 1 = medium, 2 = hard) — used to train/evaluate the router
- **gold entities** (company, amount, percent, date) — used to score extraction accuracy later, since we know exactly what was inserted into each template

**Tier definitions** (this is the actual complexity signal, not an arbitrary label):
- **Tier 0 (easy):** one sentence, one entity type, no comparison
- **Tier 1 (medium):** one sentence, 2–3 entity types (company + amount + date, etc.)
- **Tier 2 (hard):** two clauses, multiple companies, comparative/contrastive language, 4+ entities




In [ ]:
COMPANIES = ["Apple", "Tesla", "Microsoft", "Amazon", "Nvidia", "Meta", "Alphabet",
             "Netflix", "Intel", "AMD", "Salesforce", "Adobe", "PayPal", "Uber", "Shopify"]

QUARTERS = ["Q1", "Q2", "Q3", "Q4"]
YEARS = ["2023", "2024", "2025"]
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

def rand_amount():
    return round(random.uniform(1.0, 250.0), 1)

def rand_pct():
    return round(random.uniform(-15.0, 40.0), 1)

def rand_date():
    return f"{random.choice(MONTHS)} {random.randint(1,28)}, {random.choice(YEARS)}"

def gen_easy():
    c = random.choice(COMPANIES)
    amt = rand_amount()
    text = f"{c} reported revenue of ${amt} billion."
    gold = {"company": [c], "amount": [f"${amt} billion"], "percent": [], "date": []}
    return text, gold

def gen_medium():
    c = random.choice(COMPANIES)
    amt = rand_amount()
    pct = rand_pct()
    date = rand_date()
    templates = [
        (f"On {date}, {c}'s stock rose {pct}% to ${amt} after {random.choice(QUARTERS)} earnings beat estimates.",
         {"company": [c], "amount": [f"${amt}"], "percent": [f"{pct}%"], "date": [date]}),
        (f"{c} posted {random.choice(QUARTERS)} revenue of ${amt} billion, up {pct}% year over year.",
         {"company": [c], "amount": [f"${amt} billion"], "percent": [f"{pct}%"], "date": []}),
    ]
    return random.choice(templates)

def gen_hard():
    c1, c2 = random.sample(COMPANIES, 2)
    amt1, amt2 = rand_amount(), rand_amount()
    pct1, pct2 = rand_pct(), rand_pct()
    q = random.choice(QUARTERS)
    text = (f"While {c1}'s revenue grew {pct1}% YoY to ${amt1}B, beating {c2}'s {pct2}% growth "
            f"to ${amt2}B, analysts remain split on {q} guidance amid rate concerns.")
    gold = {"company": [c1, c2], "amount": [f"${amt1}B", f"${amt2}B"],
            "percent": [f"{pct1}%", f"{pct2}%"], "date": []}
    return text, gold

GENERATORS = {0: gen_easy, 1: gen_medium, 2: gen_hard}

rows = []
for tier, gen_fn in GENERATORS.items():
    for _ in range(300):
        text, gold = gen_fn()
        rows.append({"text": text, "tier": tier, "gold": gold})

df = pd.DataFrame(rows).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(df.shape)
df.groupby("tier").size()


In [ ]:
df[["text", "tier"]].sample(6, random_state=1)

## 2. Train / test split

split discipline : fit everything (embeddings are frozen/pretrained so no leakage there, but the LR router and the FAISS exemplar index both need to be fit only on `train`) on `train`, evaluate on a held-out `test` set.


In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["tier"], random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("train:", train_df.shape, "test:", test_df.shape)


## 3. Embeddings (frozen encoder)

We use `all-MiniLM-L6-v2` (22M params, runs on CPU in milliseconds) purely as a frozen feature extractor — exactly the "freeze the encoder, train a linear probe on top" pattern from the gender-probing notebooks. Nothing here gets fine-tuned.


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

train_embs = embedder.encode(train_df["text"].tolist(), batch_size=64,
                              show_progress_bar=True, normalize_embeddings=True)
test_embs = embedder.encode(test_df["text"].tolist(), batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)

print(train_embs.shape, test_embs.shape)


## 4. FAISS exemplar index (for kNN margin + density)

Embeddings are L2-normalized, so inner product == cosine similarity. We build one `IndexFlatIP` over the training embeddings and keep the tier label for every vector alongside it — this index is what signals (1) and (2) of the confidence score query at inference time.


In [ ]:
dim = train_embs.shape[1]
exemplar_index = faiss.IndexFlatIP(dim)
exemplar_index.add(train_embs.astype("float32"))
exemplar_labels = train_df["tier"].to_numpy()

print("Exemplar index size:", exemplar_index.ntotal)


### 4a. kNN vote margin

For a query embedding, retrieve its `k` nearest exemplars, tally how many belong to each tier, and take `margin = top_share - runner_up_share`.


In [ ]:
def knn_margin(query_emb, k=10):
    query_emb = np.asarray(query_emb, dtype="float32").reshape(1, -1)
    sims, idxs = exemplar_index.search(query_emb, k)
    neighbor_labels = exemplar_labels[idxs[0]]
    counts = np.bincount(neighbor_labels, minlength=3)
    shares = counts / counts.sum()
    sorted_shares = np.sort(shares)[::-1]
    margin = float(sorted_shares[0] - sorted_shares[1])
    predicted_tier_knn = int(np.argmax(shares))
    return margin, predicted_tier_knn, sims[0], idxs[0]


### 4b. Local density certainty

Mean cosine similarity to the `k` nearest neighbors, rescaled to [0,1]. Low value means the query sits in a sparse region of embedding space — the router shouldn't be confident just because its neighbors agree if those neighbors are all fairly dissimilar to begin with.


In [ ]:
def density_certainty(sims):
    # sims are cosine similarities in roughly [-1, 1] since vectors are normalized
    mean_sim = float(np.mean(sims))
    return float(np.clip((mean_sim + 1) / 2, 0.0, 1.0))


## 5. Logistic regression router

A plain multinomial logistic regression on top of the frozen embeddings — same "linear probe" methodology used for the gender-probing study, just predicting complexity tier instead of gender.


In [ ]:
router = LogisticRegression(max_iter=2000, multi_class="multinomial")
router.fit(train_embs, train_df["tier"])

test_preds = router.predict(test_embs)
print("Router test accuracy:", accuracy_score(test_df["tier"], test_preds))
print()
print(classification_report(test_df["tier"], test_preds, target_names=["easy", "medium", "hard"]))


In [ ]:
cm = confusion_matrix(test_df["tier"], test_preds)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(["easy", "medium", "hard"])
ax.set_yticks(range(3)); ax.set_yticklabels(["easy", "medium", "hard"])
ax.set_xlabel("Predicted tier"); ax.set_ylabel("True tier")
ax.set_title("Router confusion matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()


### 5a. LR predictive entropy

Turn the router's `predict_proba` output into a confidence signal: normalized entropy, inverted so **high = confident**.


In [ ]:
def entropy_confidence(proba, n_classes=3):
    proba = np.clip(proba, 1e-12, 1.0)
    h = -np.sum(proba * np.log(proba))
    h_max = np.log(n_classes)
    return float(1.0 - h / h_max)


## 6. Combined custom confidence score

Weighted sum of the three signals. Weights are a starting point (`margin` weighted highest since it's the most direct boundary signal) — worth tuning against a validation slice if you extend this further.


In [ ]:
CONF_WEIGHTS = {"margin": 0.4, "density": 0.3, "entropy": 0.3}

def compute_confidence(query_emb, k=10):
    proba = router.predict_proba(query_emb.reshape(1, -1))[0]
    lr_tier = int(np.argmax(proba))

    margin, knn_tier, sims, idxs = knn_margin(query_emb, k=k)
    density = density_certainty(sims)
    ent_conf = entropy_confidence(proba)

    confidence = (CONF_WEIGHTS["margin"] * margin +
                  CONF_WEIGHTS["density"] * density +
                  CONF_WEIGHTS["entropy"] * ent_conf)

    return {
        "lr_tier": lr_tier,
        "knn_tier": knn_tier,
        "margin": margin,
        "density": density,
        "entropy_confidence": ent_conf,
        "confidence": float(confidence),
    }

# sanity check on one test example
sample_conf = compute_confidence(test_embs[0])
sample_conf


## 7. Fallback / escalation logic

If `confidence` falls below a threshold, don't trust the router's tier prediction — bump the query up one tier instead. This is what turns "a router" into "a router with a safety net": a misrouted hard query costs latency (one tier higher), never accuracy (it never gets *downgraded* by low confidence, only escalated).


In [ ]:
CONFIDENCE_THRESHOLD = 0.55

def route_query(query_emb, threshold=CONFIDENCE_THRESHOLD):
    conf_info = compute_confidence(query_emb)
    base_tier = conf_info["lr_tier"]
    escalated = conf_info["confidence"] < threshold
    final_tier = min(base_tier + 1, 2) if escalated else base_tier
    conf_info["base_tier"] = base_tier
    conf_info["final_tier"] = final_tier
    conf_info["escalated"] = escalated
    return conf_info

# quick look at escalation rate on the test set at this threshold
decisions = [route_query(e) for e in test_embs]
esc_rate = np.mean([d["escalated"] for d in decisions])
print(f"Escalation rate at threshold={CONFIDENCE_THRESHOLD}: {esc_rate:.1%}")


## 8. Load the tiered models

Three real models, one per tier. Tier 2 is loaded 4-bit via `bitsandbytes` to keep total VRAM usage inside a free-tier T4 (16GB) alongside the other two in fp16.

| Tier | Model | Precision |
|---|---|---|
| 0 (easy) | Qwen2.5-0.5B-Instruct | fp16 |
| 1 (medium) | Qwen2.5-1.5B-Instruct | fp16 |
| 2 (hard) | Qwen2.5-3B-Instruct | 4-bit |


In [ ]:
MODEL_IDS = {
    0: "Qwen/Qwen2.5-0.5B-Instruct",
    1: "Qwen/Qwen2.5-1.5B-Instruct",
    2: "Qwen/Qwen2.5-3B-Instruct",
}

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizers, models = {}, {}
for tier, model_id in MODEL_IDS.items():
    tokenizers[tier] = AutoTokenizer.from_pretrained(model_id)
    if tier == 2:
        models[tier] = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config, device_map="auto")
    else:
        models[tier] = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map=DEVICE)
    print(f"Loaded tier {tier}: {model_id}")


## 9. Entity extraction prompt + parsing

Every tier gets the same instruction so the *only* variable between tiers is model capability, not prompt quality — otherwise the latency/accuracy comparison isn't fair.


In [ ]:
EXTRACTION_PROMPT = '''Extract financial entities from the text below. Return ONLY a JSON object with keys "company", "amount", "percent", "date" (each a list of strings, empty list if none found). No explanation, no markdown fences.

Text: {text}

JSON:'''

def extract_entities(text, tier, max_new_tokens=120):
    tok, model = tokenizers[tier], models[tier]
    prompt = EXTRACTION_PROMPT.format(text=text)
    messages = [{"role": "user", "content": prompt}]
    input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tok.eos_token_id)
    latency = time.time() - t0

    gen_text = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    parsed = parse_json_safe(gen_text)
    return parsed, latency, gen_text

def parse_json_safe(raw):
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return {"company": [], "amount": [], "percent": [], "date": []}
    try:
        obj = json.loads(match.group(0))
        for k in ["company", "amount", "percent", "date"]:
            obj.setdefault(k, [])
        return obj
    except json.JSONDecodeError:
        return {"company": [], "amount": [], "percent": [], "date": []}


## 10. Semantic cache

A second FAISS index, populated at *inference time* (starts empty), storing embeddings of queries already processed. Before routing a new query anywhere, check if a near-duplicate already exists (`cosine similarity > CACHE_THRESHOLD`) — if so, reuse its cached extraction and skip the LLM call entirely.


In [ ]:
CACHE_THRESHOLD = 0.97   # near-duplicate, not just "similar topic"

class SemanticCache:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)
        self.entries = []   # list of {"text":..., "result":..., "tier":...}

    def lookup(self, query_emb):
        if self.index.ntotal == 0:
            return None
        q = np.asarray(query_emb, dtype="float32").reshape(1, -1)
        sims, idxs = self.index.search(q, 1)
        if sims[0][0] >= CACHE_THRESHOLD:
            return self.entries[idxs[0][0]]
        return None

    def add(self, query_emb, text, result, tier):
        self.index.add(np.asarray(query_emb, dtype="float32").reshape(1, -1))
        self.entries.append({"text": text, "result": result, "tier": tier})

semantic_cache = SemanticCache(dim)


## 11. End-to-end pipeline

`cache lookup → route (LR + confidence + escalation) → extract → cache store`. Returns everything needed for evaluation: which tier actually ran, latency, whether it was a cache hit, whether it escalated.


In [ ]:
def run_pipeline(text, query_emb, threshold=CONFIDENCE_THRESHOLD):
    cached = semantic_cache.lookup(query_emb)
    if cached is not None:
        return {"result": cached["result"], "tier_used": cached["tier"],
                "latency": 0.0, "cache_hit": True, "escalated": False}

    decision = route_query(query_emb, threshold=threshold)
    final_tier = decision["final_tier"]
    result, latency, _ = extract_entities(text, final_tier)
    semantic_cache.add(query_emb, text, result, final_tier)

    return {"result": result, "tier_used": final_tier, "latency": latency,
            "cache_hit": False, "escalated": decision["escalated"]}


## 12. Evaluation

Scoring function: field-level recall against the gold entities we generated the data with (fraction of gold values that appear, as substrings, somewhere in the model's extracted values for that field — averaged across the 4 fields).

We compare three conditions on the same 60-sample test slice (kept small so the always-tier-2 baseline finishes in reasonable time on a free Colab GPU):
1. **Baseline** — every query forced to tier 2 (the "always use the big model" comparison point)
2. **Routed, no cache/no escalation** — router's raw tier prediction, straight through
3. **Full pipeline** — router + confidence-based escalation + semantic cache


In [ ]:
def field_recall(gold, pred):
    scores = []
    for field in ["company", "amount", "percent", "date"]:
        gold_vals = gold.get(field, [])
        if not gold_vals:
            continue
        pred_vals = " ".join(pred.get(field, []))
        hits = sum(1 for v in gold_vals if v.replace(" ", "") in pred_vals.replace(" ", ""))
        scores.append(hits / len(gold_vals))
    return float(np.mean(scores)) if scores else 1.0

eval_df = test_df.iloc[:60].reset_index(drop=True)
eval_embs = test_embs[:60]


In [ ]:
# --- Condition 1: baseline, always tier 2 ---
baseline_rows = []
for i, row in eval_df.iterrows():
    result, latency, _ = extract_entities(row["text"], tier=2)
    baseline_rows.append({"accuracy": field_recall(row["gold"], result), "latency": latency})
baseline_df = pd.DataFrame(baseline_rows)
print("Baseline (always tier 2) — accuracy:", baseline_df["accuracy"].mean(),
      "| mean latency:", baseline_df["latency"].mean())


In [ ]:
# --- Condition 2: routed, no escalation, no cache (raw LR tier) ---
raw_rows = []
for i, row in eval_df.iterrows():
    emb = eval_embs[i]
    tier = int(router.predict(emb.reshape(1, -1))[0])
    result, latency, _ = extract_entities(row["text"], tier=tier)
    raw_rows.append({"accuracy": field_recall(row["gold"], result), "latency": latency, "tier": tier})
raw_df = pd.DataFrame(raw_rows)
print("Routed, no escalation/cache — accuracy:", raw_df["accuracy"].mean(),
      "| mean latency:", raw_df["latency"].mean())


In [ ]:
# --- Condition 3: full pipeline (escalation + cache) ---
# run every query twice back-to-back to also exercise the cache hit path realistically
semantic_cache = SemanticCache(dim)  # reset cache for a clean run
full_rows = []
for i, row in eval_df.iterrows():
    emb = eval_embs[i]
    out = run_pipeline(row["text"], emb)
    full_rows.append({"accuracy": field_recall(row["gold"], out["result"]),
                       "latency": out["latency"], "tier": out["tier_used"],
                       "cache_hit": out["cache_hit"], "escalated": out["escalated"]})
full_df = pd.DataFrame(full_rows)
print("Full pipeline — accuracy:", full_df["accuracy"].mean(),
      "| mean latency:", full_df["latency"].mean(),
      "| escalation rate:", full_df["escalated"].mean())


In [ ]:
summary = pd.DataFrame({
    "condition": ["Baseline (always tier 2)", "Routed (no escalation/cache)", "Full pipeline"],
    "accuracy": [baseline_df["accuracy"].mean(), raw_df["accuracy"].mean(), full_df["accuracy"].mean()],
    "mean_latency_s": [baseline_df["latency"].mean(), raw_df["latency"].mean(), full_df["latency"].mean()],
})
summary["latency_reduction_vs_baseline"] = 1 - summary["mean_latency_s"] / summary["mean_latency_s"].iloc[0]
summary


### 12a. Threshold sweep — accuracy vs. latency tradeoff

Sweeping `CONFIDENCE_THRESHOLD` turns the routing decision into an actual tunable operating point rather than one fixed cutoff


In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
sweep_rows = []
for th in thresholds:
    semantic_cache = SemanticCache(dim)  # fresh cache per threshold
    accs, lats = [], []
    for i, row in eval_df.iterrows():
        emb = eval_embs[i]
        out = run_pipeline(row["text"], emb, threshold=th)
        accs.append(field_recall(row["gold"], out["result"]))
        lats.append(out["latency"])
    sweep_rows.append({"threshold": th, "accuracy": np.mean(accs), "mean_latency_s": np.mean(lats)})

sweep_df = pd.DataFrame(sweep_rows)
sweep_df


In [ ]:
fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(sweep_df["threshold"], sweep_df["accuracy"], marker="o", color="tab:blue", label="Accuracy")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Field-level accuracy", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(sweep_df["threshold"], sweep_df["mean_latency_s"], marker="s", color="tab:red", label="Mean latency (s)")
ax2.set_ylabel("Mean latency (s)", color="tab:red")
plt.title("Accuracy vs. latency across escalation thresholds")
fig.tight_layout()
plt.show()


### 12b. Ablation table

Isolates how much each upgrade (escalation, caching) contributes on its own.


In [ ]:
ablation = pd.DataFrame({
    "pipeline_variant": [
        "Always tier 2 (baseline)",
        "Router only (no escalation, no cache)",
        "Router + escalation (no cache)",
        "Router + escalation + cache (full)",
    ],
    "accuracy": [
        baseline_df["accuracy"].mean(),
        raw_df["accuracy"].mean(),
        full_df.loc[~full_df["cache_hit"], "accuracy"].mean(),  # cache misses only, isolates escalation effect
        full_df["accuracy"].mean(),
    ],
    "mean_latency_s": [
        baseline_df["latency"].mean(),
        raw_df["latency"].mean(),
        full_df.loc[~full_df["cache_hit"], "latency"].mean(),
        full_df["latency"].mean(),
    ],
})
ablation
